In [15]:
%%capture

!uv pip install transformers jinja2 tiktoken

# `inspect_ai` Workflow: Analyzing Math Reasoning Traces with `inif`

This notebook demonstrates a basic workflow integrating the `inspect_ai` library from UK AISI to facilitate interpretability analyses on eval outputs.

**Scenario:** A researcher wants to analyze at which layer depth LLMs solving math problems reach the correct number throughout reasoning, and whether there is a distinction between successful vs. unsuccessful reasoning traces.

**Model:** [GPT-OSS-20B](https://huggingface.co/openai/gpt-oss-20b), OpenAI's open-weight MoE model (21B total, 3.6B active params). The tokenization code uses a **generic chat template approach** that works with any HuggingFace tokenizer (harmony, ChatML, Kimi, etc.) — tokens are tagged by role via character-span matching.

The workflow has 5 steps:

1. Evaluate the LLM on a math benchmark (mocked here)
2. Convert the eval output to the interchange format (using the real GPT-OSS-20B tokenizer)
3. Tag tokens containing numbers
4. Extract logit lens logits for tagged positions (mocked here)
5. Separate successful and unsuccessful attempts by score

## Step 1: Evaluate the LLM on a math benchmark

In a real workflow, you'd run Inspect AI against a math benchmark:

```python
from inspect_ai import eval
from inspect_ai.dataset import hf_dataset
from inspect_ai.solver import generate
from inspect_ai.scorer import exact

log = eval(
    task=Task(
        dataset=hf_dataset("opencompass/AIME2025"),
        solver=generate(),
        scorer=exact(),
    ),
    model="openai/gpt-oss-20b",
)
```

Here we'll create a mock eval log to demonstrate the conversion.

In [16]:
from types import SimpleNamespace


def make_mock_eval_log():
    """Create a mock Inspect AI EvalLog with math reasoning samples."""

    def msg(role, content):
        return SimpleNamespace(role=role, content=content)

    def score(value, answer=None):
        return SimpleNamespace(
            value=value, answer=answer, explanation=None, metadata={}
        )

    system_prompt = (
        "Solve the following math problem. Show your reasoning step by step."
    )

    samples = [
        # Sample 0: Correct reasoning
        SimpleNamespace(
            id="aime_q1",
            messages=[
                msg("system", system_prompt),
                msg(
                    "user",
                    "Find the sum of all positive integers n "
                    "such that n^2 + 12n - 2007 is a perfect square.",
                ),
                msg(
                    "assistant",
                    "Let me work through this step by step.\n"
                    "We need n^2 + 12n - 2007 = k^2 for some non-negative integer k.\n"
                    "Completing the square: (n+6)^2 - 36 - 2007 = k^2\n"
                    "So (n+6)^2 - k^2 = 2043\n"
                    "This factors as (n+6+k)(n+6-k) = 2043\n"
                    "2043 = 3 * 681 = 3 * 3 * 227\n"
                    "The factor pairs of 2043 are: (1,2043), (3,681), (9,227)\n"
                    "For each pair (a,b) with a*b=2043 and a<=b:\n"
                    "n+6 = (a+b)/2, k = (b-a)/2\n"
                    "Pair (1,2043): n = 1022-6 = 1016, k = 1021\n"
                    "Pair (3,681): n = 342-6 = 336, k = 339\n"
                    "Pair (9,227): n = 118-6 = 112, k = 109\n"
                    "Sum = 1016 + 336 + 112 = 1464\n"
                    "The answer is 1464.",
                ),
            ],
            scores={"exact_match": score(1.0, answer="1464")},
            target="1464",
            usage=SimpleNamespace(input_tokens=45, output_tokens=280),
        ),
        # Sample 1: Incorrect reasoning (arithmetic error)
        SimpleNamespace(
            id="aime_q2",
            messages=[
                msg("system", system_prompt),
                msg(
                    "user",
                    "What is the remainder when 2^100 is divided by 7?",
                ),
                msg(
                    "assistant",
                    "I'll find the pattern of 2^n mod 7.\n"
                    "2^1 = 2 mod 7\n"
                    "2^2 = 4 mod 7\n"
                    "2^3 = 8 = 1 mod 7\n"
                    "The cycle length is 3.\n"
                    "100 = 33 * 3 + 1\n"
                    "So 2^100 = 2^1 = 2 mod 7.\n"
                    "Wait, let me recheck. 100 / 3 = 33 remainder 1.\n"
                    "Actually 33 * 3 = 99, so remainder is 1.\n"
                    "2^100 mod 7 = 2^1 mod 7 = 4.\n"
                    "The answer is 4.",
                ),
            ],
            scores={"exact_match": score(0.0, answer="4")},
            target="2",
            usage=SimpleNamespace(input_tokens=30, output_tokens=150),
        ),
        # Sample 2: Correct reasoning
        SimpleNamespace(
            id="aime_q3",
            messages=[
                msg("system", system_prompt),
                msg("user", "Compute 17 * 23."),
                msg(
                    "assistant",
                    "17 * 23 = 17 * 20 + 17 * 3 = 340 + 51 = 391.\nThe answer is 391.",
                ),
            ],
            scores={"exact_match": score(1.0, answer="391")},
            target="391",
            usage=SimpleNamespace(input_tokens=20, output_tokens=40),
        ),
    ]

    return SimpleNamespace(
        eval=SimpleNamespace(
            model="openai/gpt-oss-20b",
            task="aime_2025",
            task_version=None,
            eval_id="eval_001",
            run_id="run_001",
            inspect_version="0.3.0",
        ),
        plan=SimpleNamespace(
            config=SimpleNamespace(
                max_tokens=1024,
                temperature=0.0,
                top_p=None,
                top_k=None,
                stop_seqs=None,
                frequency_penalty=None,
                presence_penalty=None,
                seed=42,
            )
        ),
        stats=SimpleNamespace(
            started_at="2025-06-01T10:00:00+00:00",
            completed_at="2025-06-01T10:05:30+00:00",
        ),
        samples=samples,
    )


eval_log = make_mock_eval_log()
print(f"Mock eval log with {len(eval_log.samples)} samples")

Mock eval log with 3 samples


## Step 2: Convert the eval output to the interchange format

In a real workflow with an `.eval` file on disk:

```python
from inif.converters.inspect_ai import from_eval_file
doc = from_eval_file("path/to/results.eval", tokenizer="openai/gpt-oss-20b")
```

Here we convert from our mock `EvalLog` object. We pass the HuggingFace model ID `"openai/gpt-oss-20b"` as a string — `from_eval_log` loads the **real GPT-OSS-20B tokenizer** via `AutoTokenizer.from_pretrained` (a pre-instantiated tokenizer object also works). It then calls `apply_chat_template` to produce the exact token sequence the model would see (including all template delimiters, auto-generated system prompts, and role markers), runs **sequence deduplication** to extract shared token runs, and tags each token with its chat template **role** (system, user, assistant, or "template" for delimiters).

In [ ]:
from inif.converters.inspect_ai import from_eval_log

# Convert: tokenize via apply_chat_template, deduplicate, and tag chat roles
doc = from_eval_log(eval_log, tokenizer="openai/gpt-oss-20b")

print(f"InifDocument with {len(doc.samples)} samples, {len(doc.sequences)} sequence(s)")
for seq in doc.sequences:
    preview = " ".join(seq.tokens[:8]) + ("..." if len(seq.tokens) > 8 else "")
    print(f"  '{seq.id}': {seq.n_tokens} tokens — {preview}")
print(f"Model: {doc.metadata.model.name}")
print(f"Task: {doc.metadata.source_eval.task}")

In [18]:
# Inspect the first sample's structure
sample = doc.samples[0]
n_ref = sum(1 for t in sample.tokens if t.is_sequence_ref)
n_own = sum(1 for t in sample.tokens if not t.is_sequence_ref)
print(f"Sample '{sample.id}':")
print(f"  Tokens: {n_ref} sequence ref(s) + {n_own} own tokens")
print(f"  Texts: {len(sample.texts)}")
for text in sample.texts:
    preview = f"{text[:60]}..." if len(text) > 60 else text
    print(f"    {preview}")
print(f"  Scores: {[(s.scorer, s.value) for s in sample.scores]}")
print(f"  Target: {sample.target}")

# Show the token structure: ref token then first own tokens
print("\n  Token structure (first 6 entries):")
for i in range(min(6, len(sample.tokens))):
    tok = sample.tokens[i]
    if tok.is_sequence_ref:
        seq = next(s for s in doc.sequences if s.id == tok.sequence_id)
        print(
            f"    [{i:3d}] <sequence_ref '{tok.sequence_id}'> ({seq.n_tokens} tokens)"
        )
    else:
        print(f"    [{i:3d}] {repr(tok.token):25s} id={tok.id}")

Sample 'aime_q1':
  Tokens: 5 sequence ref(s) + 277 own tokens
  Texts: 3
    Solve the following math problem. Show your reasoning step b...
    Find the sum of all positive integers n such that n^2 + 12n ...
    Let me work through this step by step.
We need n^2 + 12n - 2...
  Scores: [('exact_match', 1.0)]
  Target: 1464

  Token structure (first 6 entries):
    [  0] <sequence_ref 'sequence_0'> (84 tokens)
    [  1] 'Find'                    id=11437
    [  2] ' the'                    id=290
    [  3] ' sum'                    id=4215
    [  4] ' of'                     id=328
    [  5] ' all'                    id=722


## Step 3: Tag tokens containing numbers

Since `from_eval_log` already tagged each token with its chat template role (stored as an extra field), we expand sequence references back to flat tokens and use regex tagging to mark content tokens matching `\d+` with the tag `"number"`, skipping template tokens.

In [19]:
import re
from collections import Counter

from inif import expand_sequences, select_by_tag, tag_by_predicate

# Expand sequence refs back to flat tokens (creates a working copy)
doc_expanded = expand_sequences(doc)

# Show role breakdown for first sample (roles already tagged by from_eval_log)
sample = doc_expanded.samples[0]
role_counts = Counter(getattr(t, "role", "?") for t in sample.tokens)
print(f"Sample '{sample.id}' roles:")
for role, count in sorted(role_counts.items()):
    print(f"    {role}: {count} tokens")

# Tag content tokens containing digits, skipping template tokens
for sample in doc_expanded.samples:
    tag_by_predicate(
        sample,
        lambda t: (
            t.token is not None
            and bool(re.search(r"\d+", t.token))
            and getattr(t, "role", None) != "template"
        ),
        "number",
    )

# Show what got tagged in each sample
print()
for sample in doc_expanded.samples:
    selection = select_by_tag(sample, "number")
    number_strs = [t.token for t in selection.tokens]
    print(f"Sample '{sample.id}': {len(selection.tokens)} number tokens")
    print(f"  Tokens: {number_strs[:15]}{'...' if len(number_strs) > 15 else ''}")
    print(
        f"  Positions: {selection.positions[:15]}"
        f"{'...' if len(selection.positions) > 15 else ''}"
    )
    print()

Sample 'aime_q1' roles:
    ?: 100 tokens
    assistant: 250 tokens
    template: 1 tokens
    user: 26 tokens

Sample 'aime_q1': 82 number tokens
  Tokens: ['202', '4', '06', '202', '6', '04', '01', '2', '12', '200', '7', '2', '12', '200', '7']...
  Positions: [21, 22, 24, 30, 31, 33, 35, 96, 99, 103, 104, 129, 132, 136, 137]...

Sample 'aime_q2': 52 number tokens
  Tokens: ['202', '4', '06', '202', '6', '04', '01', '2', '100', '7', '2', '7', '2', '1', '2']...
  Positions: [21, 22, 24, 30, 31, 33, 35, 90, 92, 97, 111, 116, 118, 120, 123]...

Sample 'aime_q3': 19 number tokens
  Tokens: ['202', '4', '06', '202', '6', '04', '01', '17', '23', '17', '23', '17', '20', '17', '3']...
  Positions: [21, 22, 24, 30, 31, 33, 35, 86, 89, 97, 100, 103, 106, 109, 112]...



## Step 4: Extract logit lens logits for tagged positions

In a real workflow, you'd use nnterp's high-level API with NDIF:

```python
from nnterp import StandardizedTransformer
from nnterp.interventions import logit_lens

model = StandardizedTransformer("openai/gpt-oss-20b")

results = logit_lens(model, doc_expanded, positions="number")
```

Here we mock the logit lens output to demonstrate the data storage.

In [ ]:
import random

random.seed(42)

NUM_LAYERS = 4  # Pretend the model has 4 layers for the mock

for sample in doc_expanded.samples:
    selection = select_by_tag(sample, "number")
    for token in selection.tokens:
        # Simulate logit lens: at each layer, the model's top prediction
        # For correct samples, the target number should appear earlier
        is_correct = any(s.value == 1.0 for s in sample.scores)
        logit_lens_data = {}
        for layer in range(NUM_LAYERS):
            # Mock: correct samples converge to the right token faster
            if is_correct and layer >= NUM_LAYERS // 2:
                top_token = token.token
                prob = 0.6 + 0.1 * layer
            else:
                top_token = str(random.randint(0, 9))
                prob = random.uniform(0.1, 0.4)
            logit_lens_data[f"layer_{layer}"] = {
                "top_k": [{"token": top_token, "prob": round(prob, 3)}]
            }
        token.set_extra("logit_lens", logit_lens_data)

# Show an example of stored data
example_token = select_by_tag(doc_expanded.samples[0], "number").tokens[0]
print(f"Token '{example_token.token}' (extra fields: logit_lens):")
logit_data = example_token.get_extra("logit_lens", {})
for layer, data in logit_data.items():
    top = data["top_k"][0]
    print(f"  {layer}: top prediction = '{top['token']}' (prob={top['prob']})")

## Step 5: Separate successful and unsuccessful attempts

Use score-based selection to split samples by evaluation result.

In [ ]:
from inif import filter_samples_by_score

correct = filter_samples_by_score(doc_expanded, "exact_match", lambda v: v == 1.0)
incorrect = filter_samples_by_score(doc_expanded, "exact_match", lambda v: v == 0.0)

print(f"Correct samples: {len(correct)} (IDs: {[s.id for s in correct]})")
print(f"Incorrect samples: {len(incorrect)} (IDs: {[s.id for s in incorrect]})")

# Equivalent: build a fully self-contained sub-document (sequences pruned).
correct_doc = doc_expanded.subset(
    lambda s: any(sc.scorer == "exact_match" and sc.value == 1.0 for sc in s.scores)
)
print(f"\ncorrect_doc.total_samples = {correct_doc.total_samples}")

In [ ]:
def avg_convergence_layer(samples, tag, num_layers):
    """Find the average layer where the top logit lens prediction matches."""
    convergence_layers = []
    for sample in samples:
        for token in select_by_tag(sample, tag).tokens:
            logit_data = token.get_extra("logit_lens")
            if logit_data is None:
                continue
            for layer_idx in range(num_layers):
                layer_data = logit_data.get(f"layer_{layer_idx}", {})
                top_k = layer_data.get("top_k", [])
                if top_k and top_k[0]["token"] == token.token:
                    convergence_layers.append(layer_idx)
                    break
    if not convergence_layers:
        return None
    return sum(convergence_layers) / len(convergence_layers)


avg_correct = avg_convergence_layer(correct, "number", NUM_LAYERS)
avg_incorrect = avg_convergence_layer(incorrect, "number", NUM_LAYERS)

print(f"Average convergence layer (correct):   {avg_correct:.2f}")
print(f"Average convergence layer (incorrect): {avg_incorrect}")
print()
if avg_incorrect is None:
    print("Incorrect samples never converge to the right number token.")
    print("(Expected with our mock data: wrong answers never match.)")
else:
    print(f"Difference: {avg_incorrect - avg_correct:.2f} layers")

## Serialization

The enriched document (with logit lens data embedded in tokens) can be saved and shared.

In [ ]:
from inif import load, save, to_dict, validate

# Transfer enrichment (role, tags, logit_lens) from expanded tokens back to the
# deduplicated doc, so the saved file uses sequence refs for shared token runs.
doc_enriched = doc.model_copy(deep=True)
for sample_e, sample_x in zip(doc_enriched.samples, doc_expanded.samples):
    seq_map = {s.id: s for s in doc_enriched.sequences}
    exp_idx = 0
    for tok in sample_e.tokens:
        if tok.is_sequence_ref:
            exp_idx += seq_map[tok.sequence_id].n_tokens
        else:
            exp_tok = sample_x.tokens[exp_idx]
            for key, value in exp_tok.extras.items():
                tok.set_extra(key, value)
            exp_idx += 1

# Save to disk (with sequences and enrichment)
save(doc_enriched, "aime_analysis.inif.json")
save(doc_enriched, "aime_analysis.inif")  # compressed version

# Validate the serialized output against the schema
validate(to_dict(doc_enriched))
print("Schema validation passed!")

# Reload and verify
doc_reloaded = load("aime_analysis.inif")
n_samples = len(doc_reloaded.samples)
n_seqs = len(doc_reloaded.sequences)
print(f"Reloaded: {n_samples} samples, {n_seqs} sequences")

# Verify logit lens data survived the round-trip (stored as extra field)
reloaded_expanded = expand_sequences(doc_reloaded)
reloaded_token = select_by_tag(reloaded_expanded.samples[0], "number").tokens[0]
assert reloaded_token.has_extra("logit_lens")
print(f"Logit lens data preserved for token '{reloaded_token.token}'")

In [ ]:
import json

# Preview the compact JSON structure (first sample only)
d = to_dict(doc_enriched, compact=True)
preview = {
    "metadata": d["metadata"],
    "sequences": d.get("sequences", []),
    "samples": [d["samples"][0]],
}
# Truncate tokens for display
preview["samples"][0]["tokens"] = preview["samples"][0]["tokens"][:5]
print(json.dumps(preview, indent=2))

## Appendix: Template-agnostic role tagging

The `tag_chat_roles` function from `inif.tagging` works with **any** HuggingFace chat template. Below we repeat the tokenization and role tagging with [Kimi K2.5](https://huggingface.co/moonshotai/Kimi-K2.5), which uses a completely different template format (`<|im_system|>`, `<|im_middle|>`, `<|im_end|>`, auto-injected `<think></think>` tags) — without changing any code.

In [25]:
from inif import tag_chat_roles
from inif.models import InifDocument, Metadata, ModelInfo, Sample, Token

kimi_tokenizer = AutoTokenizer.from_pretrained(
    "moonshotai/Kimi-K2.5", trust_remote_code=True
)

# Reuse the first eval sample's messages
messages = [
    {"role": m.role, "content": m.content} for m in eval_log.samples[0].messages
]

# Tokenize with apply_chat_template — same code path as GPT-OSS-20B
kimi_ids = kimi_tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=False, return_dict=False
)
kimi_tokens = [
    Token(id=tid, token=kimi_tokenizer.decode([tid], skip_special_tokens=False))
    for tid in kimi_ids
]

# Build a sample and tag roles — using the package's tag_chat_roles
kimi_sample = Sample(id="kimi_q1", tokens=kimi_tokens)
tag_chat_roles(kimi_sample, messages, kimi_tokenizer)

# Role breakdown
role_counts = Counter(getattr(t, "role", "?") for t in kimi_sample.tokens)
print(f"Kimi K2.5: {len(kimi_sample.tokens)} tokens")
for role, count in sorted(role_counts.items()):
    print(f"    {role}: {count} tokens")

# First 12 tokens — shows the different template delimiters
print("\n  Kimi template (first 12 tokens):")
for i in range(min(12, len(kimi_sample.tokens))):
    tok = kimi_sample.tokens[i]
    role = getattr(tok, "role", "?")
    print(f"    [{i:3d}] {repr(tok.token):30s} id={tok.id:<8d} role={role}")

# Show that <think></think> tags are correctly tagged as template
print("\n  Assistant boundary (think tags → template, content → assistant):")
for i, tok in enumerate(kimi_sample.tokens):
    if tok.token == "<think>":
        for j in range(i, min(i + 5, len(kimi_sample.tokens))):
            t = kimi_sample.tokens[j]
            print(f"    [{j:3d}] {repr(t.token):30s} role={getattr(t, 'role', '?')}")
        break

# Serialize the Kimi output
kimi_doc = InifDocument(
    metadata=Metadata(
        model=ModelInfo(
            name="moonshotai/Kimi-K2.5",
            revision=getattr(kimi_tokenizer, "_commit_hash", None),
        )
    ),
    samples=[kimi_sample],
)
save(kimi_doc, "kimi_analysis.inif.json")
validate(to_dict(kimi_doc))

kimi_reloaded = load("kimi_analysis.inif.json")
print(f"\nSaved and reloaded: {len(kimi_reloaded.samples[0].tokens)} tokens")
reloaded_roles = Counter(
    getattr(t, "role", "?") for t in kimi_reloaded.samples[0].tokens
)
print(f"Roles preserved: {dict(sorted(reloaded_roles.items()))}")

Kimi K2.5: 313 tokens
    assistant: 260 tokens
    system: 13 tokens
    template: 14 tokens
    user: 26 tokens

  Kimi template (first 12 tokens):
    [  0] '<|im_system|>'                id=163594   role=template
    [  1] 'system'                       id=14062    role=template
    [  2] '<|im_middle|>'                id=163601   role=template
    [  3] 'Solve'                        id=91351    role=system
    [  4] ' the'                         id=276      role=system
    [  5] ' following'                   id=4111     role=system
    [  6] ' math'                        id=15601    role=system
    [  7] ' problem'                     id=5264     role=system
    [  8] '.'                            id=13       role=system
    [  9] ' Show'                        id=12045    role=system
    [ 10] ' your'                        id=651      role=system
    [ 11] ' reasoning'                   id=57753    role=system

  Assistant boundary (think tags → template, content → assistan

In [26]:
from inif import show

show(doc_enriched)

In [27]:
!inif view kimi_analysis.inif.json

Opened /var/folders/xh/wb5mc8yn0dg3p20wq0hqtlhh0000gn/T/tmpzs2ej663.html in browser
